In [2]:
from pathlib import Path
import pandas as pd

In [3]:
PMDATA_DIR = Path("data/pmdata")

injury_frames, srpe_frames, wellness_frames = [], [], []

# p[0-9][0-9] matches p01..p16 only — NOT participant-overview.xlsx
# sorted() keeps them in order
for athlete_dir in sorted(PMDATA_DIR.glob("p[0-9][0-9]")):
    source_id = athlete_dir.name # e.g. "p01"
    pmsys = athlete_dir / "pmsys"

    # srpe + wellness are present for every athlete
    srpe = pd.read_csv(pmsys / "srpe.csv", parse_dates=["end_date_time"])
    srpe.insert(0, "source_id", source_id)
    srpe_frames.append(srpe)

    wellness = pd.read_csv(pmsys / "wellness.csv", parse_dates=["effective_time_frame"])
    wellness.insert(0, "source_id", source_id)
    wellness_frames.append(wellness)

    # guard: p08 has no injury.csv
    injury_path = pmsys / "injury.csv"
    if injury_path.exists():
        injury = pd.read_csv(injury_path, parse_dates=["effective_time_frame"])
        injury.insert(0, "source_id", source_id)
        injury_frames.append(injury)
    else:
        print(f"skipping injury for {source_id}: {injury_path} not found")

injury = pd.concat(injury_frames, ignore_index=True)
srpe = pd.concat(srpe_frames, ignore_index=True)
wellness = pd.concat(wellness_frames, ignore_index=True)

for name, df in {"injury": injury, "srpe": srpe, "wellness": wellness}.items():
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} cols, "
          f"{df['source_id'].nunique()} athletes")

participant_overview = pd.read_excel(PMDATA_DIR / "participant-overview.xlsx", header=1)

skipping injury for p08: data\pmdata\p08\pmsys\injury.csv not found
injury: 225 rows x 3 cols, 15 athletes
srpe: 783 rows x 5 cols, 16 athletes
wellness: 1747 rows x 10 cols, 16 athletes


In [4]:
participant_overview

,Participant ID,Age,Height,Gender,A or B person,Max heart rate,Date,Minutes,Seconds,Stride walk,Stride run
0,p01,48,195,male,A,182,2019-11-26 00:00:00,29,33,80.90,102.9
1,p02,60,180,male,A,169,2019-12-15 00:00:00,23,51,74.70,92.4
2,p03,25,184,male,A,157,2019-12-30 00:00:00,33,22,NaN,NaN
3,p04,26,163,female,A,195,2019-11-19 00:00:00,22,13,67.30,110.2
4,p05,35,176,male,A,184,2019-12-23 00:00:00,32,40,73.00,94.3
5,p06,42,179,male,B,181,2019-12-01 00:00:00,23,19,73.04,97.6
6,p07,26,177,male,B,NaN,2019-11-19 00:00:00,19,40,73.50,119.5
7,p08,27,186,male,B,200,2019-11-28 00:00:00,18,47,77.20,103.6
8,p09,26,180,male,B,183,2020-01-07 00:00:00,35,6,74.70,109.9
9,p10,38,179,female,B,197,2019-12-08 00:00:00,28,10,73.09,102.3


In [21]:
print(injury[injury['source_id'] == 'p08'])

    source_id             effective_time_frame injuries
0         p08 2019-11-07 06:39:48.428000+00:00       {}
1         p08 2019-11-11 13:47:05.617000+00:00       {}
2         p08 2019-11-18 08:28:53.208000+00:00       {}
3         p08 2019-11-25 08:10:11.478000+00:00       {}
4         p08 2019-12-02 08:10:19.841000+00:00       {}
..        ...                              ...      ...
220       p08 2020-01-20 15:42:00.634000+00:00       {}
221       p08 2020-01-28 20:53:03.947000+00:00       {}
222       p08 2020-02-04 21:43:29.297000+00:00       {}
223       p08 2020-02-10 15:29:39.385000+00:00       {}
224       p08 2020-02-17 16:31:30.705000+00:00       {}

[225 rows x 3 columns]
